# Zomato Restaurant Analytics — SQL Analysis

**Database:** `zomato_analysis`  
**Table:** `zomato_restaurants`  
**Engine:** MySQL 8.0.42

## Project workflow

**Python / Pandas → Data Cleaning & EDA → MySQL SQL Analysis → Power BI Dashboard**

The cleaned dataset contains **51,677 restaurant records**. Missing ratings and missing approximate costs were preserved as `NULL` during import.

> **Dataset note:** counts represent restaurant records/listings. A restaurant name can occur multiple times in the source data, so record counts should not automatically be interpreted as unique physical restaurants.


## Business questions

1. Which locations have the most restaurant records?
2. Which locations have the highest average ratings?
3. Which restaurant types are most common?
4. Which restaurant types have higher average ratings?
5. Which cuisines are most common?
6. Does online ordering relate to restaurant ratings?
7. Does table booking relate to restaurant ratings?
8. How do online ordering and table booking together relate to ratings?


## 1. Restaurant records by location

**Business purpose:** Identify the locations with the largest concentration of restaurant records.

### SQL query

```sql
SELECT
    location,
    COUNT(*) AS restaurant_count
FROM zomato_restaurants
GROUP BY location
ORDER BY restaurant_count DESC
LIMIT 10;
```


**Observed result from MySQL:**

| location              |   restaurant_count |
|:----------------------|-------------------:|
| BTM                   |               5114 |
| HSR                   |               2522 |
| Koramangala 5th Block |               2504 |
| JP Nagar              |               2234 |
| Whitefield            |               2142 |
| Indiranagar           |               2082 |
| Jayanagar             |               1926 |
| Marathahalli          |               1846 |
| Bannerghatta Road     |               1630 |
| Bellandur             |               1286 |

**Insight:** BTM has the largest number of restaurant records in the dataset with 5,114, followed by HSR and Koramangala 5th Block.

**Note:** These are record counts, not guaranteed counts of unique physical restaurants.

## 2. Average rating by location

**Business purpose:** Compare location-level ratings while excluding locations with fewer than 100 rated records.

### SQL query

```sql
SELECT
    location,
    COUNT(rate) AS rated_restaurants,
    ROUND(AVG(rate), 2) AS avg_rating
FROM zomato_restaurants
WHERE rate IS NOT NULL
GROUP BY location
HAVING COUNT(rate) >= 100
ORDER BY avg_rating DESC
LIMIT 10;
```


**Observed result from MySQL:**

| location              |   rated_restaurants |   avg_rating |
|:----------------------|--------------------:|-------------:|
| Lavelle Road          |                 485 |         4.14 |
| Koramangala 3rd Block |                 191 |         4.02 |
| St. Marks Road        |                 343 |         4.02 |
| Koramangala 5th Block |                2319 |         4.01 |
| Church Street         |                 546 |         3.99 |
| Koramangala 4th Block |                 841 |         3.92 |
| Cunningham Road       |                 475 |         3.9  |
| MG Road               |                 811 |         3.86 |
| Residency Road        |                 604 |         3.86 |
| Koramangala 7th Block |                1060 |         3.85 |

**Insight:** Among locations with at least 100 rated records, Lavelle Road has the highest average rating at 4.14. Koramangala 5th Block combines a 4.01 average with 2,319 rated records.

**Note:** The minimum-count condition reduces the influence of very small groups.

## 3. Most common restaurant types

**Business purpose:** Understand which restaurant formats dominate the dataset.

### SQL query

```sql
SELECT
    rest_type,
    COUNT(*) AS restaurant_count
FROM zomato_restaurants
GROUP BY rest_type
ORDER BY restaurant_count DESC
LIMIT 10;
```


**Observed result from MySQL:**

| rest_type          |   restaurant_count |
|:-------------------|-------------------:|
| Quick Bites        |              19115 |
| Casual Dining      |              10323 |
| Cafe               |               3730 |
| Delivery           |               2600 |
| Dessert Parlor     |               2263 |
| Takeaway, Delivery |               2034 |
| Casual Dining, Bar |               1154 |
| Bakery             |               1140 |
| Beverage Shop      |                866 |
| Bar                |                697 |

**Insight:** Quick Bites is the most common restaurant type with 19,115 records, followed by Casual Dining with 10,323.

## 4. Average rating by restaurant type

**Business purpose:** Compare ratings across restaurant types while requiring at least 100 rated records.

### SQL query

```sql
SELECT
    rest_type,
    COUNT(rate) AS rated_restaurants,
    ROUND(AVG(rate), 2) AS avg_rating
FROM zomato_restaurants
WHERE rate IS NOT NULL
GROUP BY rest_type
HAVING COUNT(rate) >= 100
ORDER BY avg_rating DESC
LIMIT 10;
```


**Observed result from MySQL:**

| rest_type                   |   rated_restaurants |   avg_rating |
|:----------------------------|--------------------:|-------------:|
| Microbrewery, Casual Dining |                 121 |         4.37 |
| Casual Dining, Cafe         |                 318 |         4.19 |
| Casual Dining, Pub          |                 127 |         4.18 |
| Fine Dining                 |                 342 |         4.15 |
| Cafe, Dessert Parlor        |                 144 |         4.15 |
| Dessert Parlor, Cafe        |                 144 |         4.14 |
| Bar, Casual Dining          |                 394 |         4.13 |
| Pub, Casual Dining          |                 236 |         4.09 |
| Casual Dining, Bar          |                1110 |         4.08 |
| Cafe, Casual Dining         |                 173 |         4.05 |

**Insight:** Among restaurant types meeting the 100-rated-record threshold, Microbrewery, Casual Dining has the highest average rating at 4.37.

**Note:** These are dataset-level associations, not evidence that a restaurant format causes higher ratings.

## 5. Most common cuisines

**Business purpose:** Identify the cuisines most frequently listed by restaurants.

### SQL query

```sql
SELECT
    TRIM(SUBSTRING_INDEX(SUBSTRING_INDEX(cuisines, ',', numbers.n), ',', -1)) AS cuisine,
    COUNT(*) AS restaurant_count
FROM zomato_restaurants
JOIN (
    SELECT 1 AS n UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4
    UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL SELECT 8
    UNION ALL SELECT 9 UNION ALL SELECT 10
) numbers
ON numbers.n <= 1 + LENGTH(cuisines) - LENGTH(REPLACE(cuisines, ',', ''))
WHERE cuisines IS NOT NULL
GROUP BY cuisine
ORDER BY restaurant_count DESC
LIMIT 15;
```


**Observed result from MySQL:**

| cuisine      |   restaurant_count |
|:-------------|-------------------:|
| North Indian |              21069 |
| Chinese      |              15534 |
| South Indian |               8638 |
| Fast Food    |               8091 |
| Biryani      |               6483 |
| Continental  |               5759 |
| Desserts     |               5631 |
| Cafe         |               5300 |
| Beverages    |               4743 |
| Italian      |               3386 |
| Bakery       |               2838 |
| Street Food  |               2593 |
| Pizza        |               2072 |
| Burger       |               2007 |
| Seafood      |               1810 |

**Insight:** North Indian is the most frequently listed cuisine with 21,069 occurrences, followed by Chinese and South Indian.

**Note:** A restaurant can list multiple cuisines, so cuisine counts can exceed the number of restaurant records.

## 6. Online ordering and ratings

**Business purpose:** Compare average ratings and average votes for restaurants with and without online ordering.

### SQL query

```sql
SELECT
    online_order,
    COUNT(rate) AS rated_restaurants,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(votes), 0) AS avg_votes
FROM zomato_restaurants
WHERE rate IS NOT NULL
GROUP BY online_order
ORDER BY avg_rating DESC;
```


**Observed result from MySQL:**

| online_order   |   rated_restaurants |   avg_rating |   avg_votes |
|:---------------|--------------------:|-------------:|------------:|
| Yes            |               27187 |         3.72 |         343 |
| No             |               14453 |         3.66 |         367 |

**Insight:** Restaurants offering online ordering have a slightly higher average rating (3.72) than restaurants without online ordering (3.66).

**Note:** This is an association in the dataset and does not establish that online ordering causes higher ratings.

## 7. Table booking and ratings

**Business purpose:** Compare restaurants that offer table booking with those that do not.

### SQL query

```sql
SELECT
    book_table,
    COUNT(rate) AS rated_restaurants,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(votes), 0) AS avg_votes
FROM zomato_restaurants
WHERE rate IS NOT NULL
GROUP BY book_table
ORDER BY avg_rating DESC;
```


**Observed result from MySQL:**

| book_table   |   rated_restaurants |   avg_rating |   avg_votes |
|:-------------|--------------------:|-------------:|------------:|
| Yes          |                6299 |         4.14 |        1172 |
| No           |               35341 |         3.62 |         206 |

**Insight:** Restaurants offering table booking have a higher average rating (4.14) than restaurants without table booking (3.62) in this dataset.

**Note:** The difference is descriptive and should not be interpreted as a causal effect of table booking.

## 8. Online ordering + table booking

**Business purpose:** Examine the combined relationship of online ordering and table booking with average ratings.

### SQL query

```sql
SELECT
    online_order,
    book_table,
    COUNT(rate) AS rated_restaurants,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(votes), 0) AS avg_votes
FROM zomato_restaurants
WHERE rate IS NOT NULL
GROUP BY online_order, book_table
ORDER BY avg_rating DESC;
```


**Observed result from MySQL:**

| online_order   | book_table   |   rated_restaurants |   avg_rating |   avg_votes |
|:---------------|:-------------|--------------------:|-------------:|------------:|
| No             | Yes          |                2549 |         4.16 |        1167 |
| Yes            | Yes          |                3750 |         4.13 |        1175 |
| Yes            | No           |               23437 |         3.66 |         210 |
| No             | No           |               11904 |         3.55 |         196 |

**Insight:** The two groups offering table booking have the highest average ratings: 4.16 without online ordering and 4.13 with online ordering. The groups without table booking have lower averages of 3.66 and 3.55.

**Note:** This comparison describes patterns in the dataset; it does not show that either service causes higher ratings.

# SQL Project Summary

### Key findings

- **Location concentration:** BTM has the largest number of restaurant records (5,114).
- **Location ratings:** Lavelle Road has the highest average rating among locations with at least 100 rated records (4.14).
- **Restaurant type concentration:** Quick Bites is the most common restaurant type (19,115 records).
- **Cuisine popularity:** North Indian is the most frequently listed cuisine, followed by Chinese and South Indian.
- **Online ordering:** Restaurants with online ordering have a slightly higher average rating (3.72 vs. 3.66).
- **Table booking:** Restaurants with table booking have a higher average rating in this dataset (4.14 vs. 3.62).
- **Combined services:** Groups offering table booking have the highest average ratings in the two-way comparison.

### SQL concepts demonstrated

`SELECT` · `WHERE` · `GROUP BY` · `ORDER BY` · `LIMIT` · `COUNT` · `AVG` · `ROUND` · `HAVING` · `IS NULL` · multi-column grouping · string splitting with `SUBSTRING_INDEX`

### Next stage

**Power BI Dashboard → Interactive visualization → Final business insights → GitHub README**
